In [1]:
from pyscf import gto, fci, mcscf
import numpy as np
from gpu4pyscf import scf


/home/mert/.local/lib/python3.10/site-packages/gpu4pyscf/lib/cutensor.py:154: UserWarning: using cupy as the tensor contraction engine.
  warnings.warn(f'using {contract_engine} as the tensor contraction engine.')


In [2]:
# Molecular Information part
# !!!!! in CI part mo not forget the change electrons 
# !!!!! also orbitals are important  

xyz = '''
N    0.654504    -1.140550     0.000000
N    1.314634    -0.001971     0.000000
N    0.661167     1.136742     0.000000
N   -0.656123     1.139570     0.000000
N   -1.314710     0.003805     0.000000
N   -0.659472    -1.137596     0.000000'''
molecule = 'N6'
spin = 0
charge =  0


In [6]:
def rhf_calculation(xyz, molecule, spin, charge):
    mol = gto.M(
          atom= xyz,
          basis = 'sto-6g',
          charge = charge,
          verbose = 5 ,
          spin = spin, 
          symmetry = True,
          output  ="./"+molecule+".txt")

    mf = mol.RHF().run()
    mf.analyze() 
    S_ij = mf.get_ovlp()
    C_ij = mf.mo_coeff
    np.savetxt("C_"+molecule+'mo_coeff.txt', C_ij,fmt = '%.8f')
    np.savetxt('S_'+molecule+'_overlap.txt', S_ij,fmt = '%.8f')

    return mf, mol 

In [4]:
def orb_occ_table(orb_energy,orb_occ):
    Homo = None
    Lumo = None
    print ("orbital  energy   occ")

    for i in range (0,(len(orb_occ))):
        print(i ,"   ", orb_energy[i],"  ",orb_occ[i])
              
    for i in range(len(orb_occ) - 1):
        if orb_occ[i] == 2  and orb_occ[i+1] == 0:
            homo = i
            lumo = i + 1
            print(f"Homo_index: {homo}, Lumo_index: {lumo}")
    return homo, lumo


In [7]:
mf, mol  = rhf_calculation( xyz , molecule, spin, charge )
homo , lumo =orb_occ_table(mf.mo_energy, mf.mo_occ)

overwrite output file: ./N6.txt
orbital  energy   occ
0     -15.758554508457339    2.0
1     -15.758544761674223    2.0
2     -15.758492780223998    2.0
3     -15.758371315440968    2.0
4     -15.758357658628817    2.0
5     -15.758278063226145    2.0
6     -1.5695246824153992    2.0
7     -1.3817505485803019    2.0
8     -1.3811305216846121    2.0
9     -1.0591657190911354    2.0
10     -1.0584585950782601    2.0
11     -0.8110338555167935    2.0
12     -0.7082484134411602    2.0
13     -0.6897779981845158    2.0
14     -0.5803722629053849    2.0
15     -0.5547822520659276    2.0
16     -0.5543945174357399    2.0
17     -0.5045468294918322    2.0
18     -0.5039399701498787    2.0
19     -0.4032593661623177    2.0
20     -0.4026635255656258    2.0
21     0.07136930211400339    0.0
22     0.0719990896397103    0.0
23     0.3042531496652843    0.0
24     0.45316810072985286    0.0
25     0.4541859621414022    0.0
26     0.634396198134376    0.0
27     0.640937236082268    0.0
28     0.64

In [8]:
n_elec= 6
n_orb = 6 # number of orbital can be determined based on the number of pi  orbitals 
casscf = mcscf.CASSCF(mf, n_orb, n_elec)
'''output in order
e_tot
e_cas
Fci vecs 
mo_coeff
mo_energy'''
mo = casscf.sort_mo([14,18,19,22,23,24])# the numbers are from the index of pi orbitals 
casscf.kernel(mo) 


(np.float64(-325.5182700062283),
 np.float64(-8.220421744165947),
 FCIvector([[ 8.90103271e-01, -2.18726557e-05, -1.09846451e-03,
             -1.25911496e-07, -1.09304835e-03,  2.14884716e-05,
              5.54188399e-05, -8.89909449e-02, -1.57400332e-07,
              4.74792309e-06, -3.40812261e-07, -1.14448991e-04,
              6.24089384e-06, -5.86928934e-08, -2.52984374e-02,
              7.95372721e-06, -3.08551412e-06,  7.94970701e-06,
              2.51509401e-02, -2.40340791e-06],
            [-2.18726557e-05, -1.95621904e-01,  5.48249077e-05,
             -9.52151364e-04,  5.55194533e-05,  4.28263133e-02,
             -4.08781957e-06, -6.12449571e-06, -1.09241107e-02,
              3.34267013e-05,  9.25506578e-04, -7.60501782e-07,
              6.43487192e-02,  1.11751671e-02, -1.47117264e-06,
             -7.25006420e-05,  3.06191589e-05, -6.72553749e-05,
              1.10898558e-06, -9.67594362e-03],
            [-1.09846451e-03,  5.48249077e-05, -2.07494885e-02,
      

In [9]:
casscf.fcisolver.max_cycle = 3000
casscf.canonicalize(mo_coeff=None, ci=None, eris=None, sort=False, cas_natorb=False, casdm1=None, verbose=3, with_meta_lowdin=False);

filename = f"{molecule}_Ci_coeffs.txt"

with open(filename, 'w', encoding='utf-8') as f:
    
    f.write("det-alpha,     det-beta,    CI coefficients\n")
    
    
    for c, ia, ib in casscf.fcisolver.large_ci(casscf.ci, n_orb, n_elec, tol=.000000001, return_strs=False):
        # Satır formatını oluşturalım
        line = f" {str(ia):<12} {str(ib):<12} {c:.15f}\n"
        
        #write in file 
        f.write(line)
        #show in terminal 
        print(line.strip())

[0 1 2]      [0 1 2]      0.890103270691473
[0 1 2]      [0 1 3]      -0.000021872655672
[0 1 2]      [0 2 3]      -0.001098464512374
[0 1 2]      [1 2 3]      -0.000000125911496
[0 1 2]      [0 1 4]      -0.001093048354055
[0 1 2]      [0 2 4]      0.000021488471571
[0 1 2]      [1 2 4]      0.000055418839851
[0 1 2]      [0 3 4]      -0.088990944897375
[0 1 2]      [1 3 4]      -0.000000157400332
[0 1 2]      [2 3 4]      0.000004747923091
[0 1 2]      [0 1 5]      -0.000000340812261
[0 1 2]      [0 2 5]      -0.000114448991459
[0 1 2]      [1 2 5]      0.000006240893843
[0 1 2]      [0 3 5]      -0.000000058692893
[0 1 2]      [1 3 5]      -0.025298437403043
[0 1 2]      [2 3 5]      0.000007953727209
[0 1 2]      [0 4 5]      -0.000003085514122
[0 1 2]      [1 4 5]      0.000007949707005
[0 1 2]      [2 4 5]      0.025150940053094
[0 1 2]      [3 4 5]      -0.000002403407913
[0 1 3]      [0 1 2]      -0.000021872655673
[0 1 3]      [0 1 3]      -0.195621903866026
[0 1 3]      [0 2 